# Context Retrieval Demo

An apartment hunting agent searches viewing notes, a rent data export, and a listings archive.

- **Part 1:** inefficient searching
- **Part 2:** too much context at once
- **Part 3:** irrelevant or stale context

# Setup

Install the packages and connect a model.

In [ ]:
%pip install -q deepagents langchain-anthropic langchain-huggingface sentence-transformers faker

In [ ]:
# @title Wrap long output lines (run this cell)
# @markdown Wraps long lines in cell output, so you don't have to scroll sideways.
from IPython.display import HTML, display

def set_css(*args):
    display(HTML('''
    <style>
      pre { white-space: pre-wrap; word-break: break-word; }
    </style>
    '''))

get_ipython().events.register('pre_run_cell', set_css)

Add `ANTHROPIC_API_KEY` in Colab's **Secrets** tab (left sidebar: key icon) and turn on notebook access.

In [ ]:
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")

# To use an OpenAI key (or another model provider) instead, just switch it:
# OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="anthropic:claude-sonnet-5",
    api_key=ANTHROPIC_API_KEY,
)

# To use a different provider instead, replace the block above, for example:
# model = init_chat_model("openai:gpt-6", api_key=OPENAI_API_KEY)

In [ ]:
# @title Helpers (run this cell)
# @markdown Two helpers keep the output short:
# @markdown - `show_steps` prints each tool call, the final answer, and the peak context
# @markdown - `peak_context` returns the most tokens the agent saw at once
from langchain_core.messages import AIMessage, ToolMessage

def peak_context(result):
    ai_messages = [m for m in result["messages"] if isinstance(m, AIMessage)]
    return max(m.usage_metadata["input_tokens"] for m in ai_messages)

def show_steps(result):
    failed = {m.tool_call_id for m in result["messages"] if isinstance(m, ToolMessage) and m.status == "error"}
    for m in result["messages"]:
        if isinstance(m, AIMessage):
            for call in m.tool_calls:
                if call["id"] in failed:
                    continue  # skip calls the model got wrong and retried
                args = str(call["args"])
                print("->", call["name"], args if len(args) < 110 else args[:110] + "...")
    print("\n" + result["messages"][-1].text)
    print(f"\n[Peak context: {peak_context(result):,} tokens]")

# The research folder

Our example is an apartment hunting agent. You're looking for a **pet-friendly 1BR under $2,500 a month** (ha. good luck in SF). The next cell builds a `research/` folder for it with made-up data:

| Path | What's in it |
|---|---|
| `notes/` | Your notes from 30 apartment viewings, one file each |
| `rent_data.csv` | Market data: median rent and number of listings by neighborhood, month, and apartment size (about 40,000 rows) |
| `listings/` | An archive of rental listings, one per file |

In [ ]:
# @title Build the research folder (run this cell)
# @markdown Writes the made-up notes, rent data, and listings described above into `research/`.
import csv
import random
import shutil
from pathlib import Path

from faker import Faker

random.seed(7)
Faker.seed(7)
fake = Faker()

ROOT = Path("research")
shutil.rmtree(ROOT, ignore_errors=True)
for sub in ["notes", "listings", "retrieved"]:
    (ROOT / sub).mkdir(parents=True)

def slug(name):
    return name.lower().replace(" ", "-").replace(",", "").replace("'", "")

def pick(sentences, n):
    return [random.choice(sentences) for _ in range(n)]

# ---- notes/: one file per apartment viewing
VIEWING = [
    "The bedroom fits a queen bed with room to spare.", "Closet space is limited.",
    "Street noise was noticeable with the windows open.", "The landlord seemed responsive.",
    "Natural light is best in the morning.", "Kitchen cabinets are a bit dated.",
    "The bathroom was recently retiled.", "There's a grocery store around the corner.",
    "The building entrance has a keypad.", "The radiators look old but work.",
    "Phone reception was weak in the bedroom.", "The hallway smelled freshly painted.",
    "Windows face a quiet courtyard.", "The stairs are steep for moving furniture.",
    "The bus stop is a five-minute walk.", "Counter space is tight.", "The fridge looks new.",
    "There's a small storage unit included.", "Water pressure in the shower was good.",
    "The neighbors upstairs were noisy during the visit.", "The oven is gas, not electric.",
    "There's a park across the street.", "The living room gets afternoon sun.",
]
LAUNDRY = ["In-unit washer and dryer.", "No laundry in the building; nearest laundromat is two blocks away.",
           "Shared laundry room on the ground floor."]
STREETS = [f"{name} {kind}" for name in ["Oak", "Pine", "Hazel", "Juniper", "Linden", "Spruce", "Aspen",
             "Magnolia", "Chestnut", "Poplar", "Sycamore", "Laurel", "Hawthorn", "Rowan", "Sequoia", "Cypress"]
           for kind in ["Street", "Avenue", "Road", "Place"]]
addresses = ["14 Maple Street"]
while len(addresses) < 30:
    address = f"{random.randint(2, 250)} {random.choice(STREETS)}"
    if address not in addresses:
        addresses.append(address)
for address in addresses:
    visited = f"2026-10-{random.randint(1, 31):02d}"
    laundry = ("Shared laundry in the basement, coin-operated, only two washers for the whole building."
               if address == "14 Maple Street" else random.choice(LAUNDRY))
    lines = [f"# Viewing notes: {address} (visited {visited})", "", "## Inside"]
    lines += [f"- {s}" for s in pick(VIEWING, 35)]
    lines += ["", "## Laundry", f"- {laundry}"]
    lines += ["", "## Neighborhood"] + [f"- {s}" for s in pick(VIEWING, 35)]
    (ROOT / "notes" / f"{slug(address)}.md").write_text("\n".join(lines))

# ---- rent_data.csv: about 40,000 rows
neighborhoods = ["Riverside"] + sorted({fake.city() for _ in range(800)} - {"Riverside"})[:209]
MONTHS = [f"{y}-{m:02d}" for y in (2024, 2025, 2026) for m in range(1, 13)][9:33]  # 2024-10 to 2026-09
with open(ROOT / "rent_data.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["neighborhood", "month", "bedrooms", "metric", "value"])
    for hood in sorted(neighborhoods):
        for month in MONTHS:
            for bedrooms in ["studio", "1BR", "2BR", "3BR"]:
                for metric in ["median_rent", "active_listings"]:
                    value = random.randint(1400, 4800) if metric == "median_rent" else random.randint(5, 300)
                    if (hood, month, bedrooms, metric) == ("Riverside", "2026-09", "1BR", "median_rent"):
                        value = 2410
                    writer.writerow([hood, month, bedrooms, metric, value])

# ---- listings/: one listing per file. The key listings are below.
BLURB = [
    "Steps from cafes, parks, and public transit.", "The kitchen has stainless steel appliances.",
    "Large windows let in plenty of light.", "Hardwood floors run throughout.",
    "The building has a bike room and package lockers.", "A short walk to the farmers market.",
    "Freshly painted with modern fixtures.", "Quiet tree-lined block with friendly neighbors.",
    "The bedroom has a generous closet.", "Heat and hot water are included.",
    "Laundry is available in the building.", "The bathroom has a deep soaking tub.",
    "The unit faces south, so it stays bright all day.", "Easy access to the highway.",
    "A shared rooftop has views of the skyline.", "Ceiling fans in every room.",
    "Lease terms are flexible.", "Close to restaurants, shops, and nightlife.",
    "The building has an elevator.", "Brand new windows keep it quiet inside.",
]
for word in ["pet", "dog", "cat", "furry", "animal"]:  # keep grep demos honest
    assert not any(word in s.lower() for s in BLURB + VIEWING), word

LISTINGS = [
    ("2025-03-04", "Charming 1BR at 14 Maple Street",
     "Rent is $2,300 per month for this 1BR at 14 Maple Street. A rare find at this price, with original details and a sunny bay window.", 220),
    ("2026-10-29", "1BR at 14 Maple Street",
     "Rent is $2,650 per month for this 1BR at 14 Maple Street.", 40),
    ("2025-02-10", "Sunny 1BR on Alder Avenue",
     "This 1BR is pet-friendly, with a small yard out back.", 150),
    ("2025-05-18", "Roomy 2BR on Elm Street",
     "This 2BR is pet-friendly and close to a large park.", 120),
    ("2026-10-20", "1BR loft on Birch Lane",
     "Small dogs and cats are welcome in this 1BR loft, and there's a dog run two blocks away.", 60),
    ("2026-10-25", "Garden 1BR on Cedar Court",
     "Bring your furry friend: this 1BR building has a grooming station and no breed restrictions.", 60),
]
FEATURES = ["a renovated kitchen", "a private balcony", "a walk-in closet", "a dishwasher",
            "a shared rooftop", "exposed brick", "central air", "a home office nook"]
for _ in range(30):
    bedrooms = random.choice(["Studio", "1BR", "2BR"])
    feature = random.choice(FEATURES)
    title = f"{bedrooms} with {feature} on {random.choice(STREETS)}"
    date = f"{random.choice([2024, 2025, 2026])}-{random.randint(1, 9):02d}-{random.randint(1, 28):02d}"
    LISTINGS.append((date, title, f"This {bedrooms.lower()} has {feature}.", random.randint(120, 200)))

for date, title, key_fact, length in LISTINGS:
    sentences = pick(BLURB, length)
    paragraphs = [key_fact] + [" ".join(sentences[i:i + 5]) for i in range(0, length, 5)]
    body = "\n\n".join(paragraphs)
    (ROOT / "listings" / f"{date}_{slug(title)}.md").write_text(f"# {title}\nPublished: {date}\n\n{body}\n")

def approx_tokens(paths):
    return sum(len(p.read_text()) for p in paths) // 4

print(f"notes/          {len(addresses):>6} files   ~{approx_tokens((ROOT / 'notes').glob('*.md')):,} tokens")
print(f"rent_data.csv   {sum(1 for _ in open(ROOT / 'rent_data.csv')) - 1:>6,} rows    ~{approx_tokens([ROOT / 'rent_data.csv']):,} tokens")
print(f"listings/       {len(LISTINGS):>6} files   ~{approx_tokens((ROOT / 'listings').glob('*.md')):,} tokens")

Every agent below uses `FilesystemBackend`, so its file tools read the real files in `research/`. The agent only sees what's inside that folder, so its paths start at `/notes/`, not `research/notes/`.

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(root_dir=ROOT, virtual_mode=True)

# 1: Inefficient searching

## Scan first, then read

Question: *What did I note about the laundry at 14 Maple Street? My viewing notes are in /notes/.*

First, an agent that opens every file in `/notes/` before answering.

In [ ]:
NOTES_QUESTION = "What did I note about the laundry at 14 Maple Street? My viewing notes are in /notes/."

agent = create_deep_agent(
    model=model,
    backend=backend,
    system_prompt="Don't use glob or grep. List /notes/ with ls, then read every file in it with read_file before answering.",
)
result_open_all = agent.invoke({"messages": [{"role": "user", "content": NOTES_QUESTION}]})

print(result_open_all["messages"][-1].text)
print(f"\n[Peak context: {peak_context(result_open_all):,} tokens]")

Now a default deep agent, which has filesystem tools built in. It can check file names with `glob` and search inside files with `grep`, then read only what matches.

In [ ]:
agent = create_deep_agent(
    model=model,
    backend=backend,
)
result_scan = agent.invoke({"messages": [{"role": "user", "content": NOTES_QUESTION}]})

show_steps(result_scan)

In [ ]:
print(f"Open every file:  {peak_context(result_open_all):>7,} tokens")
print(f"Scan first:       {peak_context(result_scan):>7,} tokens")

## grep only matches the exact words it's given

Question: *Which 1BR listings are pet-friendly?*

We tell the agent to grep for "pet-friendly".

In [ ]:
PETS_QUESTION = "Which 1BR listings are pet-friendly? Search /listings/."

agent = create_deep_agent(
    model=model,
    backend=backend,
    system_prompt="Search /listings/ with grep using the word 'pet-friendly'.",
)
result = agent.invoke({"messages": [{"role": "user", "content": PETS_QUESTION}]})

show_steps(result)

It only finds the two 2025 listings that say "pet-friendly" (one is a 2BR). There are two October 2026 listings that never use that word.

## Semantic search: matching by meaning

When you can't predict the wording, semantic search helps. The next cell splits each listing into paragraphs and loads them into a vector store, using a small embedding model that runs locally.

Like most vector stores, each piece keeps only the paragraph text. The title and date go into metadata the model never sees. That matters later.

In [ ]:
# @title Load the listings into a vector store (run this cell)
# @markdown Splits each listing into paragraphs and stores them with a small, free embedding model, so we can search by meaning.
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

# A small, free embedding model that runs locally, so no extra API key is needed.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def listing_chunks(path):
    title, published, body = path.read_text().split("\n", 2)
    return [Document(page_content=paragraph,
                     metadata={"source": path.name, "title": title[2:], "published": published.split(": ")[1]})
            for paragraph in body.strip().split("\n\n")]

chunks = [c for path in sorted((ROOT / "listings").glob("*.md")) for c in listing_chunks(path)]
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(chunks)
print(f"Indexed {len(chunks):,} paragraphs")

In [ ]:
for doc in vector_store.similarity_search("pet-friendly 1BR", k=4):
    print(f"{doc.metadata['source']:<45} {doc.page_content[:70]}...")

One search finds the Birch Lane and Cedar Court listings, even though neither uses the word "pet-friendly".

# 2: Too much context at once

## Read a page at a time

Question: *What was the median rent for a 1BR in Riverside in 2026-09?* The answer is one row in a file of about 40,000.

`read_file` never returns a whole file. It returns one page: `limit` lines (100 by default), starting at line `offset`. The first line of every result says which page the agent got and where the next one starts.

In [ ]:
agent = create_deep_agent(
    model=model,
    backend=backend,
)
result = agent.invoke({"messages": [{"role": "user", "content":
    "What was the median rent for a 1BR in Riverside in 2026-09? The data is in /rent_data.csv."
}]})

show_steps(result)
print(f"[The whole file is about {approx_tokens([ROOT / 'rent_data.csv']):,} tokens]\n")

for m in result["messages"]:
    if isinstance(m, ToolMessage) and m.name == "read_file":
        print("read_file returned:", m.content.splitlines()[0])

## Retrieve, offload, delegate

A retrieval tool takes a query, looks it up somewhere, and returns what it finds. This one returns the full text of the four best-matching listings. Each is under the 20,000-token offload limit, so all four land in the main agent's context.

In [ ]:
# @title Retrieval helpers (run this cell)
# @markdown Shared plumbing for the search tools below: find the top 4 listings, read a listing's text, and save a result to `/retrieved/`. This keeps each tool to a few lines.
from itertools import count
from langchain_core.tools import tool

def top_listings(query, filter=None):
    """Return the best-matching paragraph from each of the top 4 listings."""
    best = {}
    for doc in vector_store.similarity_search(query, k=20, filter=filter):
        best.setdefault(doc.metadata["source"], doc)  # several paragraphs can come from one listing
    return list(best.values())[:4]

def listing_text(doc):
    """The full listing, without its title or date (like the vector store's paragraphs)."""
    return (ROOT / "listings" / doc.metadata["source"]).read_text().split("\n", 2)[2]

result_number = count(1)

def save(text):
    """Save a retrieved result to /retrieved/ and return its path."""
    name = f"result_{next(result_number)}.md"
    (ROOT / "retrieved" / name).write_text(text)
    return f"/retrieved/{name}"

In [ ]:
@tool
def search_listings(query: str) -> str:
    """Search rental listings by meaning. Returns the full text of the 4 best-matching listings."""
    return "\n\n---\n\n".join(listing_text(doc) for doc in top_listings(query))

LISTINGS_QUESTION = "Find pet-friendly 1BR apartments. Give me 3 bullet points."

In [ ]:
agent = create_deep_agent(
    model=model,
    tools=[search_listings],
    backend=backend,
)
result_payload = agent.invoke({"messages": [{"role": "user", "content": LISTINGS_QUESTION}]})

show_steps(result_payload)

That works, but all four full listings land in the main agent's context, including everything it doesn't need. So we split the work in two:

- **Retrieve and offload:** `search_listings_to_files` runs the same search, but saves each listing to a file in `/retrieved/` and returns only the file paths.
- **Delegate:** the main agent hands each path to a `chunk-analyst` subagent. The subagent reads the listing in its own context window and sends back just the key facts.

The main agent only ever sees four short paths and four short summaries.

In [ ]:
@tool
def search_listings_to_files(query: str) -> str:
    """Search rental listings by meaning. Saves the 4 best-matching listings to files and returns their paths."""
    return "\n".join(save(listing_text(doc)) for doc in top_listings(query))

chunk_analyst = {
    "name": "chunk-analyst",
    "description": "Reads one retrieved file and returns its key facts. Give it the file path.",
    "system_prompt": "Read the file you're given with read_file. Return the key facts in 2 or 3 short bullet points. Don't return the raw text.",
}

agent = create_deep_agent(
    model=model,
    tools=[search_listings_to_files],
    subagents=[chunk_analyst],
    backend=backend,
    system_prompt="Search with search_listings_to_files. Don't read the files yourself: send each path to chunk-analyst, then combine what it returns.",
)
result_paths = agent.invoke({"messages": [{"role": "user", "content": LISTINGS_QUESTION}]})

show_steps(result_paths)

In [ ]:
print(f"Return full listings:           {peak_context(result_payload):>7,} tokens")
print(f"Return paths + chunk-analyst:   {peak_context(result_paths):>7,} tokens")

# 3: Irrelevant or stale context

## Put the date in the text

Question: *Is the 1BR at 14 Maple Street under my $2,500 budget?*

The archive has a long 2025 listing and a short 2026 relisting, with different rents. Our search tool returns listing text only, so neither file says when it was published.

In [ ]:
BUDGET_QUESTION = "Is the 1BR at 14 Maple Street under my $2,500/month budget? One sentence."

result = agent.invoke({"messages": [{"role": "user", "content": BUDGET_QUESTION}]})
show_steps(result)

Now the search tool writes a short header at the top of each file, and can skip older listings with `published_after`. The subagent returns the date along with the key facts and flags anything outdated.

In [ ]:
@tool
def search_listings_dated(query: str, published_after: str = "") -> str:
    """Search rental listings by meaning. Saves the 4 best-matching listings to files and returns their paths. Set published_after (YYYY-MM-DD) to skip older listings."""
    paths = []
    for doc in top_listings(query, filter=lambda d: d.metadata["published"] > published_after):
        header = f"{doc.metadata['title']}\npublished: {doc.metadata['published']}\n\n"  # date moves into the text
        paths.append(save(header + listing_text(doc)))
    return "\n".join(paths)

dated_analyst = {
    **chunk_analyst,
    "system_prompt": "Read the file you're given with read_file. Return the key facts in 2 or 3 short bullet points, plus the published date. Flag anything that looks outdated.",
}

agent = create_deep_agent(
    model=model,
    tools=[search_listings_dated],
    subagents=[dated_analyst],
    backend=backend,
    system_prompt="Search with search_listings_dated. Don't read the files yourself: send each path to chunk-analyst, then combine what it returns. Prefer the most recent information.",
)
result = agent.invoke({"messages": [{"role": "user", "content": BUDGET_QUESTION}]})

show_steps(result)

# Wrap-up

- **Inefficient searching:** scan with `glob` and `grep` before reading. When the wording is predictable, have the agent try synonyms. When it isn't, add a semantic search tool.
- **Too much context at once:** grep first, then read a page at a time with `offset` and `limit`. Have retrieval tools return paths instead of full text.
- **Irrelevant or stale context:** let a subagent read each result in its own context, and put the date in the text so the model can tell old from new.

To learn more, see the [LangChain Academy Deep Agents course](https://academy.langchain.com/courses/foundation-introduction-to-deepagents).